# Iterators and Generators in Python

Iterators are one of the core ideas behind Python's looping system. They allow data to be accessed one element at a time and make Python efficient when working with sequences, files, streams, and large datasets. This notebook begins with iterators and then continues into generators.


[Jump to Generators](#generator-section)


## Learning Goals

By the end of this notebook, you should be able to:

- Define what an iterator is.
- Explain the difference between an iterable and an iterator.
- Understand the iterator protocol using `__iter__()` and `__next__()`.
- See how `for` loops use iterators internally.
- Build a custom iterator class.
- Identify iterator exhaustion and common iterator mistakes.
- Understand what a generator is and how `yield` turns a function into a generator.


## 1. Introduction

An iterator is an object that allows sequential access to elements of a collection one item at a time, without exposing the underlying structure.

Python uses iterators internally for many operations such as:

- `for` loops
- Traversing `list`, `tuple`, `set`, and `dict`
- Generators
- File reading
- Many functions in `itertools`


### Why Iterators Matter

Iterators are important because they:

- Enable lazy evaluation, where values are produced only when needed.
- Allow large datasets to be processed without loading everything into memory.
- Provide a standard protocol used across Python.
- Form the foundation for generators and streaming pipelines.
- Make it possible to create custom iterable objects.


## 2. Core Idea (Intuition)

Imagine a playlist of songs.

There are two possible ways to work with it:

| Approach | Description |
|---|---|
| List | Load all songs into memory |
| Iterator | Ask for the next song only when needed |

An iterator behaves like this:

- Next song -> play
- Next song -> play
- Next song -> play
- Stop when no songs remain

This makes iterators both lazy and memory-efficient.


## 3. The Iterator Protocol

Python defines a strict protocol for iterators.

An object is an iterator if it implements these two methods:

1. `__iter__()`
2. `__next__()`

### `__iter__()`

Returns the iterator object itself.

### `__next__()`

Returns the next value.

If no values remain, Python must raise `StopIteration`.

### Visual Model


In [2]:
print("Iterable")
print("   |")
print("   | iter()")
print("   v")
print("Iterator")
print("   |")
print("   | next()")
print("   v")
print("Next Value")


Iterable
   |
   | iter()
   v
Iterator
   |
   | next()
   v
Next Value


## 4. Iterables vs Iterators

| Feature | Iterable | Iterator |
|---|---|---|
| Can be looped | Yes | Yes |
| Has `__iter__()` | Yes | Yes |
| Has `__next__()` | No | Yes |
| Example | `list`, `tuple`, `dict` | result of `iter()` |

Example:


| Concept  | Analogy    | Meaning                                            |
| -------- | ---------- | -------------------------------------------------- |
| Iterable | A book     | A collection you *can start reading from*          |
| Iterator | A bookmark | Keeps track of **where you are currently reading** |


#### What is an Iterator?

An iterator is an object that:

1. Remembers its position

2. Returns the next item

3. Stops when finished

It has two important methods:

`__iter__()`

`__next__()`

In [ ]:
# if somthing is itrable it needs to have spcial method also called dunder method __iter__
# we can check this using inbuilt dir() function
a = [1,2,34,]

print(dir(a))


In [6]:
numbers = [1, 2, 3]

print(numbers)          # iterable
print(iter(numbers))    # iterator


[1, 2, 3]


In [ ]:
#  Inspect iter()
numbers = [1, 2, 3]

it = iter(numbers)

print(type(it))
print(dir(it)) # it has bother __iter__ and __next__ methods that's why its an itrator
print(dir(numbers)) # while and itrable will only have __iter__ method

<class 'list_iterator'>
['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__length_hint__', '__lt__', '__ne__', '__new__', '__next__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__']


In [ ]:
numbers = [10, 20, 30]

it = iter(numbers) # an itrator remembers its state

print(next(it))
print(next(it))
print(next(it))
print(next(it)) # if run out of values it raises stop itration exception
# but when we run for loop on itrable like list it knows how to handle the exception and stops


10
20
30


StopIteration: 

In [ ]:
# Example of for loop itrating over an itrable
a = [1,2,3,4,5]

for i in a:
    print(i)

1
2
3
4
5


In [ ]:
# how for loop works internally
#Python converts for loop  internally into:
a = [1,2,3,4,5]
i_a = iter(a)

while True:
    try:
       i = next(i_a)
       print(i)
    except StopIteration:
        break



1
2
3
4
5


In [ ]:
# each itrator tarcks its own progress
a = [10,20,30]

it1 = iter(a)
it2 = iter(a)

print(next(it1))
print(next(it1))

print(next(it2))

10
20
10


In [ ]:
# An itrator object is coceptually similar to

class ListIterator:

    def __init__(self, data):
        self.data = data
        self.index = 0

    def __next__(self):
        if self.index >= len(self.data):
            raise StopIteration

        value = self.data[self.index]
        self.index += 1
        return value

#     So the iterator remembers position using internal state (usually an index or pointer).

# It is not predicting anything. It just reads the next stored position.

#### What Does `__iter__()` Return?

For an iterator, `__iter__()` returns itself.

Conceptually:
```python
def __iter__(self):
    return self
```


In [21]:
# Example:

it = iter([1,2,3])

print(iter(it) is it)



True


Output:

True

Reason: An iterator is already ready to iterate.

So it returns itself.

In [ ]:
# exmaple to show iterators are forward only they exhust once finished
# you have to create new iterator if you wanna start somthings again



a = [10,20,30]

it = iter(a)

print(next(it))
print(next(it))

print(list(it))

print(next(it))

print(list(it))

10
20
[30]


StopIteration: 

In [ ]:
numbers = [10, 20, 30]

it = iter(numbers) # an itrator remembers its state

print(next(it))
print(next(it))
print(next(it))


10
20
30


In [ ]:
numbers = [10, 20, 30]

it = iter(numbers) # an itrator remembers its state

print(next(it))
print(next(it))


10
20


`numbers` is an iterable. Calling `iter(numbers)` creates an iterator from it.


## 5. Creating an Iterator from an Iterable


In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))
print(next(iterator))


Expected output:

```text
10
20
30
```

Explanation:

- `numbers` is an iterable.
- `iter(numbers)` returns an iterator.
- `next(iterator)` returns the next value each time it is called.


## 6. `StopIteration`

When an iterator runs out of values, Python raises `StopIteration`.


In [ ]:
numbers = [1, 2]

it = iter(numbers)

print(next(it))
print(next(it))

try:
    print(next(it))
except StopIteration:
    print("StopIteration")


Expected output:

```text
1
2
StopIteration
```

This exception is Python's way of saying: no more elements are available.


## 7. How a `for` Loop Uses Iterators Internally

A `for` loop is syntactic sugar.


In [ ]:
numbers = [1, 2, 3]

for n in numbers:
    print(n)


Equivalent low-level version:


In [ ]:
numbers = [1, 2, 3]

iterator = iter(numbers)

while True:
    try:
        value = next(iterator)
        print(value)
    except StopIteration:
        break


Key insight: a `for` loop is built from `iter()`, repeated `next()` calls, and automatic `StopIteration` handling.


## 8. Example 1: Iterating a List


In [ ]:
data = ["A", "B", "C"]

it = iter(data)

print(next(it))
print(next(it))
print(next(it))


Expected output:

```text
A
B
C
```


## 9. Example 2: Iterating a String


In [ ]:
text = "AI"

it = iter(text)

print(next(it))
print(next(it))


Expected output:

```text
A
I
```


## 10. Example 3: Iterating a Dictionary


In [ ]:
data = {"a": 1, "b": 2}

it = iter(data)

print(next(it))
print(next(it))


Expected output:

```text
a
b
```

Dictionary iteration returns keys by default.


## 11. Example 4: File Iterator

Files are iterators, which means they can be read one line at a time.


In [ ]:
with open("example.txt", "w", encoding="utf-8") as file:
    file.write("First line\nSecond line\nThird line\n")

with open("example.txt", "r", encoding="utf-8") as file:
    it = iter(file)
    print(next(it))
    print(next(it))


This reads the file lazily, line by line, instead of loading the entire file into memory at once.

In this notebook, the example creates a small file first so the cell runs cleanly.


## 12. Example 5: Using `next()` with a Default

You can provide a default value to prevent `StopIteration` from being raised.


In [ ]:
numbers = [1]

it = iter(numbers)

print(next(it, "END"))
print(next(it, "END"))


Expected output:

```text
1
END
```

If the iterator has no values left, Python returns the default value instead.


## Creating Custom Iterators

A custom iterator class gives you full control over how values are produced.


## 13. Example: Custom Iterator Class


In [ ]:
class Counter:
    def __init__(self, max_value):
        self.max = max_value
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.max:
            raise StopIteration

        self.current += 1
        return self.current


In [26]:
nums = [1,2,3]

it = iter(nums)

print(next(it))
print(next(it))

nums.append(4)

print(next(it))
print(next(it))

1
2
3
4


Using the iterator:


In [ ]:
counter = Counter(3)

for num in counter:
    print(num)


Expected output:

```text
1
2
3
```


## 14. Memory Advantage of Iterators

Compare these two approaches.


### List


In [ ]:
numbers = [x for x in range(1_000_000)]


This creates one million values in memory immediately.


### Iterator


In [ ]:
numbers = iter(range(1_000_000))


This produces values only when requested, so memory usage stays much smaller.


## 15. Making Custome itrable classes

In [7]:
n = range(1,10)

for i in n:
    print(i) # it does the itration cuse it contains __iter__ method


# print(next(n)) #it's not an itrable but not an iterator cuse it doesn't contain __next__ method of itself


1
2
3
4
5
6
7
8
9


In [ ]:
class Range:

    def __init__(self, start, stop=None, step=1):

        if stop is None:
            start, stop = 0, start

        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):

        if self.current >= self.stop:
            raise StopIteration

        value = self.current
        self.current += self.step
        return value

2
4
6


## Common Mistakes

### Mistake 1: Iterator Exhaustion


In [ ]:
numbers = [1, 2, 3]

it = iter(numbers)

for n in it:
    print(n)

for n in it:
    print(n)


Expected output:

```text
1
2
3
```

The second loop prints nothing because the iterator has already been consumed.


### Mistake 2: Forgetting `StopIteration`


In [ ]:
class BadIterator:
    def __init__(self):
        self.current = 1

    def __iter__(self):
        return self

    def __next__(self):
        return self.current


This causes infinite iteration because `__next__()` never raises `StopIteration`.


## Mental Model to Remember

- Iterable = container
- Iterator = cursor moving through the container
- `next()` = move the cursor forward
- `StopIteration` = no more elements


## 16. Moving from Iterators to Generators

Now that you understand iterators, the next topic is generators. Generators follow the same idea of producing values one at a time, but Python gives us a simpler way to build them.


<a id="generator-section"></a>

## 17. What is a Generator?

A generator is a special type of function that produces values one at a time instead of returning them all at once.

Normal functions:

- run completely
- return one value
- then terminate

Generators:

- pause execution
- remember their state
- resume later

The keyword that makes this possible is `yield`.

When Python sees `yield`, it transforms the function into a generator function.
